In [51]:
from openai import OpenAI
openai_client = OpenAI()

In [52]:
from dotenv import load_dotenv
load_dotenv()

True

In [53]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [54]:
question = 'I just discovered the course. Can I join now?'
answer = llm(question)
print(answer)

Absolutely — you can likely join now, but it depends on the course’s enrollment rules.

If you’re wondering whether it’s too late:
- **Open-enrollment/self-paced course:** you can usually join anytime.
- **Live cohort course:** you may still be able to join if the instructor allows late entry.
- **Course with a deadline/waitlist:** you might need special permission.

If you want, I can help you draft a short message to the instructor asking whether late enrollment is possible.


In [55]:
context = """I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

edit on GitHub
#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

edit on GitHub
#What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram and Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.

Don’t post questions in chat as they may be missed if the room is very active."""

In [56]:
prompt = f'''Your task is to answer questions from the course participants based on the provided context. 

Use the context to find relevant informaitona nd provide acurate answers. 

If the answer is not found in the context, respond with "I do not know"

Question: 
{question}

Context:
{context}'''

In [57]:
print(prompt)

Your task is to answer questions from the course participants based on the provided context. 

Use the context to find relevant informaitona nd provide acurate answers. 

If the answer is not found in the context, respond with "I do not know"

Question: 
I just discovered the course. Can I join now?

Context:
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

edit on GitHub
#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

edit on GitHub
#What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
The zoom link is only published to instructors

In [58]:
question = 'I just discovered the course. Can I join now?' # copying from above just so we have it clearly all in one place
answer = llm(prompt)
print(answer)

Yes, you can still join. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [59]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [60]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

courses_raw

[{'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 48},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 460},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 397},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 153},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 249}]

In [61]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1400

In [74]:
"""
We start with R in RAG, which is Retrieval, but he calls it SEARCH.
"""

from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [63]:
question = 'I just discovered the course. Can I join now?' # copying from above just so we have it clearly all in one place
index.search(question)  # returns questions from all of the courses, llm_zoomcamp, AI Dev Zoomcamp, Data Engineering Zoomcamp, etc.

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '9e508f2212',
  'course': 'data-engineering-zoomcamp',
 

In [64]:
index.search(question, filter_dict={'course':'llm-zoomcamp'})  # returns all the questions only from the llm-zoomcamp course

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

In [65]:
search_results =index.search(question, filter_dict={'course':'llm-zoomcamp'}, num_results=5) # if we just want 5 results

In [71]:
# this was our original RAG function
# def rag(question):
#     search_results = search(question)
#     user_prompt = build_prompt(question, search_results)
#     return llm(user_prompt)

# We have now covered the search function, so let's add it
def search(question, course='llm-zoomcamp'):
    boost_dict={'question':2.0, 'section':0.5}
    filter_dict={'course':course}
    
    return index.search(question,
                        boost_dict=boost_dict,
                        filter_dict=filter_dict,
                        num_results=5)

In [ ]:
search_results = search(question)  # note that this search() function is the minsearch's Index() function and not
                                   # the search() function we defined just above.
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

In [ ]:
"""
We move on to the A in RAG, which is Augmented, but he calls it PROMPT.


He said that Prompt is split into two parts: 

1) Instructions 
2) User prompt

So let's split up what we had towards the beginnig of all of this.
"""

In [ ]:
INSTRUCTIONS = f'''Your task is to answer questions from the course participants based on the provided context. 

Use the context to find relevant informaitona nd provide acurate answers. 

If the answer is not found in the context, respond with "I do not know"
'''

In [ ]:
USER_PROMPT_TEMPLATE = '''
Question: 
{question}

Context:
{context}
'''